# 5. Memory, and the options only CUDA has

A gradient needs the forward wave field again, but backwards in time: from the last time step to the first.
PyFWI keeps snapshots of the forward field as it goes (the *checkpoints*, a share `chpr` of the steps) and
rebuilds the field between them during the gradient. The two backends keep them in different places:

* **OpenCL** copies every checkpoint to the memory of the computer as the forward modelling makes it, and back
  to the GPU during the gradient, through the connection between the two;
* **CUDA** keeps them in the memory of the GPU when they fit (in 80 % of its free memory), without any copy; the
  rest goes to the memory of the computer, as on OpenCL.

So CUDA is faster, and uses the memory of the GPU instead. It also has two options to use much less of it:
*boundary saving* and *FP16*. This notebook measures all of it. It takes about a minute.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from comparison import (  # the code shared by the notebooks of this folder: comparison.py
    CUDA_FIXED,
    OPENCL_FIXED,
    check_engines,
    colour,
    cuda_options,
    gpu_memory_peak,
    inclusion_model,
    name,
    relative_difference,
    style,
    survey,
    time_forward_and_gradient,
)

check_engines()

## Settings

A 300 × 300 model (7 m cells), 2 sources, 1.4 s of recording: about two gigabytes of checkpoints. `REPEATS`: each time
is the median of this many runs, after one run that is not timed.

In [ ]:
NZ, NX, NS, T = 300, 300, 2, 1.4
REPEATS = 2

## Where the checkpoints live

The same forward modelling and gradient on both backends, in the `fixed` mode. The memory of the GPU is measured
from the memory pool of CuPy, which only the CUDA backend uses: for OpenCL it is not measured here.

In [ ]:
problem = survey(NZ, NX, NS, T)
true, start = inclusion_model(NZ, NX), inclusion_model(NZ, NX, smooth=1)

checkpoints = {}
for engine in (OPENCL_FIXED, CUDA_FIXED):
    with gpu_memory_peak() as memory:
        run = checkpoints[engine] = time_forward_and_gradient(problem, engine, true, start, REPEATS)
    on_gpu = engine == CUDA_FIXED
    print(f"{name(engine):15s} forward {run['forward']:6.2f} s   gradient {run['gradient']:6.2f} s   "
          f"checkpoints {run['checkpoint_mb']:6.0f} MB in the memory of the {'GPU' if on_gpu else 'computer'}"
          + (f"   GPU memory at its peak {memory['mb']:6.0f} MB" if on_gpu else ""))

## The options only CUDA has

Both are environment variables, read when a propagator is created (`cuda_options` sets them for a block of
code); both need the CUDA backend in the `fixed` mode.

* **Boundary saving** (`PYFWI_BOUNDARY_SAVING=1`): instead of whole snapshots every few steps, the forward
  modelling stores, at every step, a thin band of the field around the edges of the model; the gradient rebuilds
  the field backwards in time from that band. Much less memory, a little more computation.
* **FP16** (`PYFWI_FP16=1`): the wave fields are stored as half-precision numbers, 16 bits instead of 32. Half the
  memory, and faster, because there is half as much memory to read; but every number keeps about 3 significant
  digits instead of 7.

Each configuration runs the same forward modelling and gradient; its results are compared with those of the
default configuration, as a relative difference and as a signal-to-noise ratio in decibels (dB): 40 dB means a
difference of 1 %, 20 dB of 10 %, 0 dB a difference as large as the result itself.

In [ ]:
OPTIONS = {  # name: (boundary saving, FP16)
    "default": (False, False),
    "boundary saving": (True, False),
    "FP16": (False, True),
    "both": (True, True),
}
options = {}
for option, (boundary_saving, fp16) in OPTIONS.items():
    with cuda_options(boundary_saving, fp16), gpu_memory_peak() as memory:
        run = time_forward_and_gradient(problem, CUDA_FIXED, true, start, REPEATS)
    options[option] = dict(run, memory_mb=memory["mb"])

def compared(difference):
    return "identical" if difference == 0 else f"{difference:.1e} ({-20 * np.log10(difference):.0f} dB)"


default = options["default"]["last"]
print(f"{'':16s}{'GPU memory':>12s}{'forward':>10s}{'gradient':>10s}   difference from the default")
for option, run in options.items():
    seismograms = relative_difference(run["last"]["seismograms"], default["seismograms"])
    gradient = relative_difference(run["last"]["gradient"], default["gradient"])
    run["difference"] = {"seismograms": seismograms, "gradient": gradient}
    print(f"{option:16s}{run['memory_mb']:9.0f} MB{run['forward']:8.3f} s{run['gradient']:8.3f} s   "
          f"seismograms {compared(seismograms)}, gradient {compared(gradient)}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
names = list(options)
x = np.arange(len(names))
panels = [
    ("Peak GPU memory (MB, lower is better)", [options[o]["memory_mb"] for o in names], "{:.0f} MB", "linear"),
    ("Gradient (s, lower is better)", [options[o]["gradient"] for o in names], "{:.3f} s", "linear"),
    ("Difference of the gradient from the default\n(log scale)",
     [max(options[o]["difference"]["gradient"], 1e-9) for o in names], "{:.0e}", "log"),
]
for ax, (title, values, text, scale) in zip(axes, panels):
    style(ax)
    ax.bar(x, values, 0.6, color=colour(CUDA_FIXED))
    ax.set_yscale(scale)
    for xi, value in zip(x, values):
        label = text.format(value) if names[xi] != "default" or scale == "linear" else "reference"
        ax.text(xi, value, label, ha="center", va="bottom", fontsize=9)
    ax.set_xticks(x, names)
    ax.set_title(title, loc="left", fontsize=10)
axes[2].set_ylim(1e-9, 1)
fig.suptitle(f"The options only CUDA has, {NZ} x {NX} model, {NS} sources", x=0.01, ha="left")
fig.tight_layout()
plt.show()

## What this says

* **CUDA trades copies for GPU memory.** OpenCL keeps the checkpoints in the memory of the computer and pays
  for the copies at every checkpoint; CUDA keeps them on the GPU when they fit. On a GPU with less memory, or for
  a larger model, part of them go to the memory of the computer, and CUDA copies them too.
* **Boundary saving takes several times less GPU memory** (4.2× to 10.7× in the final document): the band
  grows with the edges of the model and the checkpoints with its area, so the gain grows with the size of the model
  and with `chpr`. Its gradient costs a little more computation, and differs from the default only by rounding.
* **FP16 halves the memory and is faster, but it is much less accurate**: here the seismograms differ from those
  of single precision by about a percent, and the gradient by tens of percents. The gradient comes from the
  residual, the part of the data the model does not explain yet, which is much weaker than the waves themselves;
  the rounding of half precision, relative to the waves, covers much of it. The tolerances of release 0.4.0 (40 dB
  on the seismograms, 30 dB on the gradients) were measured on test models of 40 × 40 points; they do not hold on
  this model. Take FP16 as a way to look at waves with little memory, not for gradients or inversions.
* **Both together** give the smallest memory, with the accuracy of FP16. OpenCL has neither option: they rely on
  the kernels written for CUDA.

The [README](../../README.md#tuning-the-cuda-backend) and the [user guide](../../docs/cuda-migration/user-guide.md)
list every option of the CUDA backend.